## Project brief
Analyze a synthetic delivery dataset using NumPy only. You will validate data, filter invalid records, compute descriptive statistics, identify outliers using an IQR rule, and produce a concise report.

In [1]:
import numpy as np

rng = np.random.default_rng(2026)
n = 2_000
order_id = np.arange(1, n + 1)
distance_km = rng.uniform(0.5, 25, size=n)
delivery_minutes = 12 + 2.1 * distance_km + rng.normal(0, 7, size=n)
late_minutes = np.maximum(delivery_minutes - 45, 0)
# Inject a few invalid values for the cleaning stage.
delivery_minutes[[17, 83]] = np.nan
delivery_minutes[[101]] = -1

print('records:', order_id.size)
print('distance shape:', distance_km.shape)
print('delivery time shape:', delivery_minutes.shape)

records: 2000
distance shape: (2000,)
delivery time shape: (2000,)


In [2]:
# 1. Validate and clean
valid = np.isfinite(distance_km) & np.isfinite(delivery_minutes) & (distance_km >= 0) & (delivery_minutes >= 0)
clean_ids = order_id[valid]
clean_distance = distance_km[valid]
clean_minutes = delivery_minutes[valid]
print('valid records:', clean_ids.size)
print('rejected records:', n - clean_ids.size)

# 2. Descriptive statistics
summary = {
    'mean_minutes': float(clean_minutes.mean()),
    'median_minutes': float(np.median(clean_minutes)),
    'std_minutes': float(clean_minutes.std(ddof=1)),
    'p90_minutes': float(np.percentile(clean_minutes, 90)),
    'min_minutes': float(clean_minutes.min()),
    'max_minutes': float(clean_minutes.max()),
}
summary

valid records: 1992
rejected records: 8


{'mean_minutes': 39.32135656054379,
 'median_minutes': 39.16778888579681,
 'std_minutes': 16.288052151195224,
 'p90_minutes': 60.94125980381655,
 'min_minutes': 0.13869908330489622,
 'max_minutes': 79.91429418186759}

In [3]:
# 3. IQR-based outlier flagging (a descriptive heuristic, not automatic proof of bad data).
q1, q3 = np.percentile(clean_minutes, [25, 75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outlier_mask = (clean_minutes < lower) | (clean_minutes > upper)
print('Q1 / Q3 / IQR:', q1, q3, iqr)
print('outlier bounds:', lower, upper)
print('outlier count:', outlier_mask.sum())

# 4. SLA analysis
sla_minutes = 45
within_sla = clean_minutes <= sla_minutes
print('within SLA:', int(within_sla.sum()))
print('SLA success rate:', float(within_sla.mean()))
print('average distance (km):', float(clean_distance.mean()))

Q1 / Q3 / IQR: 26.21249804334022 52.54051640244687 26.32801835910665
outlier bounds: -13.279529495319757 92.03254394110684
outlier count: 0
within SLA: 1218
SLA success rate: 0.6114457831325302
average distance (km): 12.97458277219698


In [4]:
# 5. Segment the data by distance and summarize each segment.
bins = np.array([0, 5, 10, 15, 20, 25.01])
segment = np.digitize(clean_distance, bins[1:-1])
for group_id in np.unique(segment):
    group = clean_minutes[segment == group_id]
    print(f'segment {group_id}: n={group.size}, mean={group.mean():.2f} min, median={np.median(group):.2f} min')

segment 0: n=339, mean=18.17 min, median=18.82 min
segment 1: n=421, mean=27.92 min, median=27.93 min
segment 2: n=385, mean=37.90 min, median=37.43 min
segment 3: n=433, mean=49.19 min, median=49.57 min
segment 4: n=414, mean=59.24 min, median=59.35 min


In [5]:
# 6. Save a reproducible cleaned dataset and report.
from pathlib import Path
import json

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)
clean_matrix = np.column_stack([clean_ids, clean_distance, clean_minutes])
np.save(output_dir / 'clean_deliveries.npy', clean_matrix)
with open(output_dir / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print('saved:', output_dir / 'clean_deliveries.npy')
print('saved:', output_dir / 'summary.json')

saved: ..\data\processed\clean_deliveries.npy
saved: ..\data\processed\summary.json


## Final deliverables
- Explain the cleaning rule and rejected-row count.
- Report mean, median, standard deviation, and 90th percentile.
- Report SLA success rate and distance-segment summaries.
- Discuss limitations: the data is synthetic, and IQR flags should be investigated rather than blindly deleted.
- Extend the project with CSV input, reusable functions, and a benchmark for the cleaning transformation.